# VAE z32 β=1 normal-only — ejecución individual posterior

Normal-only con la separación ya utilizada por AE z32. El AE z32 terminado no
se reentrena ni se utiliza como inicialización. Los checkpoints nuevos nacen de
pesos y Adam nuevos; test permanece cerrado. Cada notebook de entrenamiento
contiene un único experimento y `RUN_TRAINING=False` por defecto.

Primero AE z64. Después, abrir cada notebook VAE por separado, de uno en uno.
No hay una cola ni un botón para entrenar los cuatro modelos. Las rutas de
salida son nuevas por experimento y timestamp; los históricos se conservan.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, subprocess, sys, zipfile
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")


In [ ]:
# @title Paquete y fuentes existentes
ZIP_SOURCE = "Upload" # @param ["Upload", "Drive"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab_ae64_vae_normal_only.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_remaining_normal_models" # @param {type:"string"}
DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
LOCAL_PROCESSED_DIR = "/content/workshop4_remaining_data/processed" # @param {type:"string"}
OUTPUT_PARENT = "/content/drive/MyDrive/Workshop_4/remaining_normal_models" # @param {type:"string"}
AE_Z32_CHECKPOINT = "/content/drive/MyDrive/Workshop_4/normal_only_ae/ae_z32_normal_only_20261116T134404_817936Z/best_inference.pt" # @param {type:"string"}
REFERENCE_PROTOCOL = "/content/drive/MyDrive/Workshop_4/normal_only_ae/development_20261116T134404_817936Z/frozen_protocol.json" # @param {type:"string"}
BEST8_CHECKPOINT = "/content/drive/MyDrive/Workshop_4/runs/workshop4_clipping_v2_prueba12/ae_z32/checkpoints/best.pt" # @param {type:"string"}
DEVICE = "cuda" # @param ["cuda", "cpu"]
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Subir exactamente el ZIP nuevo AE z64/VAE")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
with zipfile.ZipFile(zip_path) as archive:
    helpers = [name for name in archive.namelist() if name.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("ZIP de proyecto incompatible")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import extract_project_zip, stage_processed_parquet, runtime_info
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
assert (WORKSHOP_DIR/"src/train_remaining_normal.py").is_file()
assert (WORKSHOP_DIR/"src/evaluate_normal_models.py").is_file()
# Reutilizar copias locales idénticas; solo se copian train/val, nunca test.
PROCESSED_DIR = stage_processed_parquet(DRIVE_PROCESSED_DIR, LOCAL_PROCESSED_DIR)
print(runtime_info())


## Referencia común antes de gastar GPU

Se utiliza el `frozen_protocol.json` asociado al `.sha256` confirmado por el
usuario. Se verifican existencia, SHA, filas y fuente AE z32; no se inspecciona
su Adam. Las 7.640 entradas de desarrollo y las entradas de calibración se
reutilizan por ID, orden y hash. Si falta una entrada o cambia el contenido,
la evaluación falla sin rellenar ni generar otra muestra silenciosamente.

Las etiquetas siguen siendo normales cuando y=espacio (ID1), clase modal de
train. El resto son anomalías proxy; la prevalencia agregada ronda el 81%.
Scores altos=NLL/MSE mayor=anomalía. AP se interpreta frente a prevalencia.
El valor `Test ready` anterior no autoriza abrir test en estos notebooks.


In [ ]:
from src.evaluate_normal_models import read_reference_protocol
reference_path = Path(REFERENCE_PROTOCOL)
reference_sha_path = reference_path.with_suffix(".sha256")
if not reference_path.is_file() or not reference_sha_path.is_file():
    raise FileNotFoundError("Se necesitan frozen_protocol.json y frozen_protocol.sha256 de desarrollo")
reference, reference_sha = read_reference_protocol(reference_path)
reference_model = reference["models"].get("normal_only") or reference["models"].get("ae_z32")
if reference_model is None:
    raise ValueError("La referencia no contiene el AE z32 normal-only")
if reference_model.get("epoch") != 14 or reference_model.get("global_updates") != 60928:
    raise ValueError("La referencia no coincide con el mejor AE z32 comunicado: época14/update60928")
if len(reference["selected_rows"]["evaluation"]) != 7640:
    raise ValueError("La referencia no contiene las 7.640 filas confirmadas")
if reference.get("normal_class") != 1:
    raise ValueError("Cambió la clase normal respecto al protocolo confirmado")
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b""):
            h.update(block)
    return h.hexdigest()
if not Path(AE_Z32_CHECKPOINT).is_file():
    raise FileNotFoundError(AE_Z32_CHECKPOINT)
if file_sha256(AE_Z32_CHECKPOINT) != reference_model["sha256"]:
    raise ValueError("El checkpoint AE z32 difiere del que fijó la referencia")
print("Referencia verificada:", reference_sha)
print("Calibración:", len(reference["selected_rows"]["calibration"]), "Desarrollo:", 7640)
print("Test cerrado. No se reentrena AE z32.")


Antes de iniciar GPU se comprueba también que los datos actuales resuelvan
las filas y hashes de la referencia. Es una comprobación de lectura de train/val,
sin scores ni actualización de modelos. Una referencia válida como JSON no
basta si cambió el contenido de los datos.


In [ ]:
import time
from src.data import DataPipeline, load_config
from src.normal_only_protocol import role_partition, build_train_input_guard
from src.evaluate_normal_models import resolve_reference_rows
data_config = load_config(WORKSHOP_DIR/"config.json")
data_config["processed_dir"] = Path(PROCESSED_DIR)
preflight_pipeline = DataPipeline(data_config)
preflight_deadline = time.monotonic() + 600
preflight_partition = role_partition(preflight_pipeline, reference["config"]["seed"])
preflight_guard = build_train_input_guard(preflight_pipeline, reference["normal_class"], deadline=preflight_deadline)
preflight_rows = resolve_reference_rows(preflight_pipeline, reference, preflight_partition,
                                       reference["normal_class"], preflight_guard, preflight_deadline)
assert len(preflight_rows["evaluation"]) == 7640
print("Datos, roles y cohorte exacta verificados antes del entrenamiento.")
del preflight_rows, preflight_guard, preflight_pipeline


## Entrenamiento único desde cero

Arquitectura recurrente existente: embedding 32, hidden 128, una capa, dropout0.
AE z64 cambia solo el ancho del cuello de botella solicitado. Los VAE usan la
clase variacional existente con z32, μ/logvar y reparametrización. No se añaden
tanh latente, annealing KL, free-bits, teacher forcing ni nuevas conexiones.

Mismo train normal, particiones y filtros exactos de entradas; clipping FP64,
guardas por paso, mejor sano y checkpoints inmutables. Adam lr 0.0003, batch 128,
máximo30 épocas/150.000 updates/4 h, evaluación cada 512 y al terminar época.
Early stopping después de tres épocas mínimas y cinco evaluaciones sin mejora.

VAE: train minimiza reconstrucción MUESTREADA + β·KL, registradas por separado.
Selección/early stopping usa reconstrucción determinista en μ, el mismo criterio
de reconstrucción que los AE. NLL(μ)+βKL se registra como objetivo en μ;
**no es ELBO Monte Carlo ni log-verosimilitud marginal**. Las evaluaciones no
consumen el RNG que muestrea el entrenamiento. No se cambia β durante un run.


In [ ]:
# @title Solo vae_z32_b1; ejecución desactivada
EXPERIMENT_ID = "vae_z32_b1"
RUN_TRAINING = False # @param {type:"boolean"}
TRAIN_DIR = Path(OUTPUT_PARENT)/f"{EXPERIMENT_ID}_{run_id}"
config_path = WORKSHOP_DIR/"config/remaining_models"/f"{EXPERIMENT_ID}.json"
training_config = json.loads(config_path.read_text())
display(training_config)
assert training_config["experiment_id"] == EXPERIMENT_ID
train_command = [sys.executable, "-m", "src.train_remaining_normal", "--config", str(config_path),
                 "--processed-dir", str(PROCESSED_DIR), "--device", DEVICE, "--output-dir", str(TRAIN_DIR)]
subprocess.run(train_command, cwd=WORKSHOP_DIR, check=True, timeout=30)
print("Salida nueva:", TRAIN_DIR)


In [ ]:
training_summary = None
if RUN_TRAINING:
    info = runtime_info()
    if DEVICE == "cuda" and not info["cuda_available"]:
        raise RuntimeError("Seleccionar runtime GPU T4 antes de entrenar")
    completed = subprocess.run(train_command + ["--execute"], cwd=WORKSHOP_DIR,
                               check=False, timeout=training_config["max_seconds"]+60)
    print("Código de salida:", completed.returncode)
    if (TRAIN_DIR/"summary.json").is_file():
        training_summary = json.loads((TRAIN_DIR/"summary.json").read_text())
        print(training_summary["status"], training_summary.get("reason"))
        print("Mejor exportado:", training_summary.get("best_inference"))
    else:
        print("Ejecución parcial; conservar checkpoints. No se relanza automáticamente.")
else:
    print("Solo plan. Ningún entrenamiento fue iniciado.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
if RUN_TRAINING and (TRAIN_DIR/"history.csv").is_file():
    history = pd.read_csv(TRAIN_DIR/"history.csv")
    display(history.tail())
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    history.plot(x="global_updates", y=["train_reconstruction_nats_per_sequence",
                 "selection_reconstruction_nats_per_sequence"], ax=axes[0])
    history.plot(x="global_updates", y=["train_kl_nats_per_sequence",
                 "selection_kl_nats_per_sequence"], ax=axes[1])
    history.plot(x="global_updates", y="swap_damage_nats_per_sequence", ax=axes[2])
    for ax, title in zip(axes, ("Reconstrucción normal", "KL (solo VAE)", "Utilidad de z/μ por intercambio")):
        ax.set_title(title)
        ax.grid(alpha=.2)
    plt.tight_layout()
    plt.show()


## Comparación en el desarrollo ya fijado

Se cargan el nuevo mejor y el AE z32 terminado. Opcionalmente se añaden otros
modelos ya entrenados y best8 exploratorio. El comparador no entrena y no tiene
modo test. Todos comparten las filas exactas de la referencia; cada modelo y
score calibra un umbral propio con los mismos normales de calibration.

Comunes: MSE categórica y NLL. VAE: KL separado; NLL+KL (coeficiente1) como score
secundario común entre β, y NLL+βKL claramente identificado como alternativa de
entrenamiento. No se suma KL a MSE ni se elige un score mirando test. Las
métricas en desarrollo no seleccionan un ganador automáticamente.


In [ ]:
RUN_DEVELOPMENT_EVALUATION = True # @param {type:"boolean"}
COMPARE_BEST8 = False # @param {type:"boolean"}
EXISTING_TRAINING_DIR = "" # @param {type:"string"}
# Opcional: incluir checkpoints ya terminados, nunca configuraciones para entrenar.
OTHER_COMPLETED_MODELS_JSON = "{}" # @param {type:"string"}
candidate_dir = Path(EXISTING_TRAINING_DIR) if EXISTING_TRAINING_DIR.strip() else TRAIN_DIR
candidate = None
if (candidate_dir/"summary.json").is_file():
    training_summary = json.loads((candidate_dir/"summary.json").read_text())
    if training_summary.get("best_inference"):
        candidate = Path(training_summary["best_inference"]["path"])
COMPARISON_DIR = Path(OUTPUT_PARENT)/f"comparison_{EXPERIMENT_ID}_{run_id}"
if candidate is not None and candidate.is_file():
    checkpoints = {"ae_z32": AE_Z32_CHECKPOINT, EXPERIMENT_ID: str(candidate)}
    extra = json.loads(OTHER_COMPLETED_MODELS_JSON)
    if not isinstance(extra, dict) or set(extra) & set(checkpoints):
        raise ValueError("Los adicionales deben ser un objeto JSON sin reemplazar AE z32 o el modelo actual")
    checkpoints.update(extra)
    if COMPARE_BEST8:
        checkpoints["best8"] = BEST8_CHECKPOINT
    command = [sys.executable, "-m", "src.evaluate_normal_models", "--reference-protocol", REFERENCE_PROTOCOL,
               "--processed-dir", str(PROCESSED_DIR), "--device", DEVICE, "--output-dir", str(COMPARISON_DIR)]
    for label, path in checkpoints.items():
        if not Path(path).is_file():
            raise FileNotFoundError(path)
        command += ["--checkpoint", label + "=" + path]
    subprocess.run(command, cwd=WORKSHOP_DIR, check=True, timeout=30)
    if RUN_DEVELOPMENT_EVALUATION:
        completed = subprocess.run(command + ["--execute"], cwd=WORKSHOP_DIR, check=False, timeout=1860)
        print("Código de salida:", completed.returncode)
        if (COMPARISON_DIR/"summary.json").is_file():
            result = json.loads((COMPARISON_DIR/"summary.json").read_text())
            print(result["status"], result.get("reason"))
            print("Filas comunes:", result.get("evaluation_examples"), "Referencia exacta:", result.get("exact_reference_reused"))
            print("Test permanece cerrado:", not result.get("test_accessed", False))
else:
    print("Entrenar solo este modelo o indicar su EXISTING_TRAINING_DIR; AE z32 no se reentrena.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
def show_common_comparison(folder):
    folder = Path(folder)
    if not (folder/"metrics.json").is_file():
        return
    payload = json.loads((folder/"metrics.json").read_text())
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    for label, scores in payload["models"].items():
        for score, metric in scores.items():
            rows.append({"modelo": label, "score": score, "n": metric["examples"],
                         "prevalencia": metric["anomaly_prevalence"], "ROC-AUC": metric["roc_auc"],
                         "AP": metric["average_precision"], "PR-AUC trapecio": metric["pr_auc_trapezoidal"],
                         "precision": metric["precision"], "recall": metric["recall"], "F1": metric["f1"],
                         "FPR": metric["false_positive_rate"], "confusión": metric["confusion"]})
            # Curvas comunes de reconstrucción. Los scores VAE+KL son secundarios.
            if score not in ("brier_mse", "nll_nats_per_sequence"):
                continue
            roc, pr = metric["roc_curve"], metric["precision_recall_curve"]
            if roc:
                axes[0].plot([r["fpr"] for r in roc], [r["tpr"] for r in roc], label=f"{label}/{score}")
            if pr:
                axes[1].plot([0]+[r["recall"] for r in pr], [1]+[r["precision"] for r in pr], label=f"{label}/{score}")
    display(pd.DataFrame(rows))
    axes[0].plot([0, 1], [0, 1], "k--", lw=.7)
    axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC de desarrollo")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="PR de desarrollo; AP separada")
    for ax in axes:
        ax.grid(alpha=.2)
        ax.legend(fontsize=6)
    plt.tight_layout()
    plt.show()
    if (folder/"reconstruction_latent.json").is_file():
        display(json.loads((folder/"reconstruction_latent.json").read_text())["evaluation"])

show_common_comparison(COMPARISON_DIR)


Conservar history.csv, updates.jsonl, checkpoints, best_inference.pt,
summary.json y la configuración por experimento en Drive. Cada comparación
produce scores.csv, metrics.csv/JSON, cohort.json, reconstruction_latent.json y
frozen_protocol.json con SHA. Test sigue cerrado aunque existan umbrales.
Reportar épocas/updates realmente completados y el motivo de parada; un corte
no equivale a estabilidad de 30 épocas. Consultar REMAINING_NORMAL_MODELS.md.
